# Build synthetic component maps (Planck 6-channel, Nside=4096)

Populate `/rds/rds-lxu/flamingo/integrated_maps_synthetic/components/` with
beam-**unconvolved** maps at the six Planck HFI frequencies from
`reference_tables/planck_info.png` (Table I):

| GHz | 100 | 143 | 217 | 353 | 545 | 857 |
|-----|-----|-----|-----|-----|-----|-----|

**Components are stored separately — not summed.** Beam convolution,
coaddition, and noise will be applied in a later step.

Each component is in $\mu\mathrm{K}_\mathrm{CMB}$ (or Compton $y$ / Doppler
$b$ / Jy/sr for the native FLAMINGO products) at native
$N_\mathrm{side}=4096$, beam-unconvolved.

| Component | Source | Action |
|-----------|--------|--------|
| CMB | FLAMINGO $\kappa$ + CAMB $C_\ell^{TT}$ | simulate (pixell lensing) |
| CIB | Yang26 released bands 217/353/545/857 | copy intensity FITS; 100/143 GHz via greybody SED |
| tSZ | `lensed_tSZ_rot.fits` (Compton $y$) | copy $y$; build $\Delta T(\nu)=T_\mathrm{CMB}\,y\,f(\nu)$ |
| kSZ | `lensed_kSZ_rot.fits` (Doppler $b$) | copy $b$; $\Delta T=-T_\mathrm{CMB}\,b$ (freq.-indep.) |

Input maps: `/rds/rds-lxu/flamingo/integrated_maps_synthetic/L2800N5040/HYDRO_FIDUCIAL/lightcone0_shells`.

Requires `pip install -e ".[cmb]"` for the lensed CMB step (`camb`, `pixell`).

In [1]:
from pathlib import Path

from flamingo_mock import MockConfig
from flamingo_mock.config import PLANCK_FREQUENCIES_GHZ
from flamingo_mock import cib, cmb, ksz, tsz

cfg = MockConfig(
    frequencies=PLANCK_FREQUENCIES_GHZ,
    nside=4096,
    seed=42,
)
cfg.make_dirs()

OUT = cfg.out_dir / "components"
for sub in ("cmb", "cib", "tsz", "ksz"):
    (OUT / sub).mkdir(parents=True, exist_ok=True)

print("data:", cfg.data_dir)
print("out: ", OUT)
print("freqs:", list(cfg.frequencies))
print("Nside:", cfg.nside)

data: /rds/rds-lxu/flamingo/integrated_maps_synthetic/L2800N5040/HYDRO_FIDUCIAL/lightcone0_shells
out:  /rds/rds-lxu/flamingo/integrated_maps_synthetic/components
freqs: [100.0, 143.0, 217.0, 353.0, 545.0, 857.0]
Nside: 4096


## 1. Lensed primary CMB

In [2]:
# ~30–60 min at Nside=4096; skipped if the FITS already exists.
cmb_uK = cmb.make_lensed_cmb(cfg, out_dir=OUT / "cmb")
print(f"CMB lensed: std={cmb_uK.std():.2f} uK")

CMB: reusing cached /rds/rds-lxu/flamingo/integrated_maps_synthetic/components/cmb/primary_CMB_T_lensed_nside4096_seed42.fits
CMB lensed: std=107.72 uK


## 2. CIB — copy released bands, approximate 100/143 GHz

Released lensed bandpass maps (217/353/545/857 GHz) are copied from the
FLAMINGO tree. **100 and 143 GHz** are outside the released set; we build
them with the three-parameter greybody SED at $z_\mathrm{eff}=1.5$ (same
method as `flamingo_mock.cib.approximate_cib_intensity`).

Note: `CIB_nonrot_BANDPASS_F143_three_params.fits` exists but is **not**
lensed — we do not use it.

In [3]:
# Archive released intensity maps [Jy/sr] (symlink to save space)
cib.copy_released_cib_intensity(cfg, out_dir=OUT / "cib", use_symlink=True)

# Thermodynamic maps [uK_CMB] at all six frequencies
cib_uK = cib.make_cib_maps(cfg, out_dir=OUT / "cib")

CIB: reusing CIB_I_217GHz_nside4096.fits
CIB: reusing CIB_I_353GHz_nside4096.fits
CIB: reusing CIB_I_545GHz_nside4096.fits
CIB: reusing CIB_I_857GHz_nside4096.fits
CIB: loading released 217 GHz...
CIB: loading released 353 GHz...
CIB: loading released 545 GHz...
CIB: loading released 857 GHz...
  wrote /rds/rds-lxu/flamingo/integrated_maps_synthetic/components/cib/CIB_deltaT_100GHz_nside4096.fits (0.81 GB)
  CIB 100 GHz: std=4.094e+00 uK | SED scale from 217 GHz at z_eff=1.5 (x0.074)
  wrote /rds/rds-lxu/flamingo/integrated_maps_synthetic/components/cib/CIB_deltaT_143GHz_nside4096.fits (0.81 GB)
  CIB 143 GHz: std=8.777e+00 uK | SED scale from 217 GHz at z_eff=1.5 (x0.251)
  wrote /rds/rds-lxu/flamingo/integrated_maps_synthetic/components/cib/CIB_deltaT_217GHz_nside4096.fits (0.81 GB)
  CIB 217 GHz: std=2.749e+01 uK | released 217 GHz
  wrote /rds/rds-lxu/flamingo/integrated_maps_synthetic/components/cib/CIB_deltaT_353GHz_nside4096.fits (0.81 GB)
  CIB 353 GHz: std=1.709e+02 uK | relea

## 3. tSZ — copy Compton-$y$, build $\Delta T(\nu)$

The lensed Compton-$y$ map is archived from `lensed_tSZ_rot.fits`. Per-frequency
temperature maps use the non-relativistic spectral function
$f(x)=x\coth(x/2)-4$.

In [4]:
tsz.archive_compton_y(cfg, out_dir=OUT / "tsz", use_symlink=True)
tsz_uK = tsz.make_tsz_maps(cfg, out_dir=OUT / "tsz")

tSZ: reusing compton_y_nside4096.fits
tSZ: loading lensed Compton-y map...
tSZ: y mean=1.6109e-06, std=2.0692e-06
  wrote /rds/rds-lxu/flamingo/integrated_maps_synthetic/components/tsz/compton_y_nside4096.fits (0.81 GB)
  wrote /rds/rds-lxu/flamingo/integrated_maps_synthetic/components/tsz/tSZ_deltaT_100GHz_nside4096.fits (0.81 GB)
  tSZ 100 GHz: std=8.505e+00 uK
  wrote /rds/rds-lxu/flamingo/integrated_maps_synthetic/components/tsz/tSZ_deltaT_143GHz_nside4096.fits (0.81 GB)
  tSZ 143 GHz: std=5.867e+00 uK
  wrote /rds/rds-lxu/flamingo/integrated_maps_synthetic/components/tsz/tSZ_deltaT_217GHz_nside4096.fits (0.81 GB)
  tSZ 217 GHz: std=4.385e-02 uK
  wrote /rds/rds-lxu/flamingo/integrated_maps_synthetic/components/tsz/tSZ_deltaT_353GHz_nside4096.fits (0.81 GB)
  tSZ 353 GHz: std=1.264e+01 uK
  wrote /rds/rds-lxu/flamingo/integrated_maps_synthetic/components/tsz/tSZ_deltaT_545GHz_nside4096.fits (0.81 GB)
  tSZ 545 GHz: std=3.157e+01 uK
  wrote /rds/rds-lxu/flamingo/integrated_maps_synt

## 4. kSZ — copy Doppler-$b$, convert to $\mu\mathrm{K}_\mathrm{CMB}$

The lensed kSZ map (`lensed_kSZ_rot.fits`) stores Doppler $b$ with
$\Delta T/T_\mathrm{CMB}=-b$. The thermodynamic map is frequency independent.

In [ ]:
ksz.archive_doppler_b(cfg, out_dir=OUT / "ksz", use_symlink=True)
ksz_uK = ksz.make_ksz_map(cfg, out_dir=OUT / "ksz")
print(f"kSZ dT: std={ksz_uK.std():.3e} uK")

kSZ: reusing doppler_b_nside4096.fits
kSZ: loading lensed Doppler-b map...
kSZ: b mean=-1.3004e-07, std=1.5236e-06
  wrote /rds/rds-lxu/flamingo/integrated_maps_synthetic/components/ksz/doppler_b_nside4096.fits (0.81 GB)


## 5. Inventory

In [ ]:
print(f"\nProducts under {OUT}:\n")
for sub in sorted(OUT.iterdir()):
    if sub.is_dir():
        files = sorted(sub.glob("*.fits"))
        print(f"  {sub.name}/  ({len(files)} files)")
        for p in files:
            print(f"    {p.name}  ({p.stat().st_size/1e9:.2f} GB)")

## 6. Publication-quality Mollweide maps

Maps are rendered at native $N_\mathrm{side}=4096$ (no downgrade).

Figures are saved to `../figures/` as PDF + PNG.

In [ ]:
import io

import healpy as hp
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

FIG_DIR = Path("../figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

NSIDE_VIZ = 4096  # native map resolution for mollview

mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "mathtext.fontset": "cm",
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "axes.linewidth": 1.0,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "legend.frameon": False,
})


def savefig(fig, name: str, *, tight: bool = False) -> None:
    kw = {"pad_inches": 0.08}
    if tight:
        kw["bbox_inches"] = "tight"
    fig.savefig(FIG_DIR / f"{name}.pdf", **kw)
    fig.savefig(FIG_DIR / f"{name}.png", **kw)
    print(f"Wrote {FIG_DIR / name}.pdf")


def load_for_viz(path: Path) -> np.ndarray:
    m = hp.read_map(str(path), dtype=np.float64)
    if hp.get_nside(m) != NSIDE_VIZ:
        m = hp.ud_grade(m, NSIDE_VIZ)
    return m


def render_mollview_tile(
    m,
    title,
    unit,
    *,
    symmetric=True,
    pct=(1, 99),
    cmap="RdBu_r",
) -> Image.Image:
    """Render one full-sky map to a PNG tile (avoids healpy multi-panel bugs)."""
    x = np.asarray(m, dtype=np.float64)
    plt.close("all")
    if symmetric:
        vmax = np.percentile(np.abs(x[np.isfinite(x)]), pct[1])
        hp.mollview(x, min=-vmax, max=vmax, title=title, unit=unit,
                    cmap=cmap, xsize=700, hold=False)
    else:
        vmin, vmax = np.percentile(x[np.isfinite(x)], pct)
        hp.mollview(x, min=vmin, max=vmax, title=title, unit=unit,
                    cmap=cmap, xsize=700, hold=False)
    hp.graticule(dpar=30, dmer=60, alpha=0.35)
    buf = io.BytesIO()
    plt.gcf().savefig(buf, format="png", dpi=200, bbox_inches="tight", pad_inches=0.06)
    plt.close("all")
    buf.seek(0)
    return Image.open(buf).convert("RGB")


def mollview_gallery(panels, nrows, ncols, figsize_scale, save_name, suptitle=None):
    """Stitch per-map mollview tiles into a publication-quality figure."""
    tiles = [render_mollview_tile(**p) for p in panels]
    tw, th = tiles[0].size
    gap = 24
    canvas = Image.new(
        "RGB",
        (ncols * tw + (ncols - 1) * gap, nrows * th + (nrows - 1) * gap),
        "white",
    )
    for idx, tile in enumerate(tiles):
        r, c = divmod(idx, ncols)
        canvas.paste(tile, (c * (tw + gap), r * (th + gap)))
    w, h = canvas.size
    title_pad = 0.35 if suptitle else 0.0
    fig = plt.figure(
        figsize=(w / 200 * figsize_scale, h / 200 * figsize_scale + title_pad),
        dpi=200,
    )
    ax = fig.add_axes([0, 0, 1, 0.93 if suptitle else 1])
    ax.imshow(canvas)
    ax.axis("off")
    if suptitle:
        fig.suptitle(suptitle, y=0.995, fontsize=14)
    savefig(fig, save_name, tight=False)
    plt.show()


def mollview_single(m, title, unit, save_name, **kw):
    """Single-panel mollview."""
    tile = render_mollview_tile(m, title, unit, **kw)
    fig, ax = plt.subplots(figsize=(tile.size[0] / 200, tile.size[1] / 200), dpi=200)
    ax.imshow(tile)
    ax.axis("off")
    savefig(fig, save_name, tight=True)
    plt.show()

print(f"viz Nside={NSIDE_VIZ}, fig dir={FIG_DIR.resolve()}")

### 6.1 Lensed primary CMB

In [ ]:
cmb_path = OUT / "cmb" / f"primary_CMB_T_lensed_nside{cfg.nside}_seed{cfg.seed}.fits"
mollview_single(
    load_for_viz(cmb_path),
    title=rf"Lensed primary CMB ($N_{{\mathrm{{side}}}}={NSIDE_VIZ}$)",
    unit=r"$\mu\mathrm{K}_\mathrm{CMB}$",
    save_name="components_cmb_lensed_mollview",
)

### 6.2 tSZ — Compton $y$ and $\Delta T(\nu)$

Compton-$y$ is dimensionless; per-frequency panels use symmetric scales set
by the 99th percentile of $|\Delta T|$ at each band.

In [ ]:
mollview_single(
    load_for_viz(OUT / "tsz" / f"compton_y_nside{cfg.nside}.fits"),
    title=r"Compton $y$ (lensed tSZ)",
    unit="$y$",
    save_name="components_tsz_compton_y_mollview",
    pct=(0.5, 99.5),
)

freqs = list(cfg.frequencies)
mollview_gallery(
    panels=[
        dict(
            m=load_for_viz(OUT / "tsz" / f"tSZ_deltaT_{nu:.0f}GHz_nside{cfg.nside}.fits"),
            title=rf"tSZ $\Delta T$ at ${nu:.0f}\,\mathrm{{GHz}}$",
            unit=r"$\mu\mathrm{K}_\mathrm{CMB}$",
            symmetric=True,
        )
        for nu in freqs
    ],
    nrows=2,
    ncols=3,
    figsize_scale=1.0,
    save_name="components_tsz_deltaT_allfreq_mollview",
    suptitle=rf"Thermal SZ temperature maps (beam-unconvolved, $N_{{\mathrm{{side}}}}={NSIDE_VIZ}$)",
)

### 6.3 kSZ — Doppler $b$ and thermodynamic map

In [ ]:
mollview_gallery(
    panels=[
        dict(
            m=load_for_viz(OUT / "ksz" / f"doppler_b_nside{cfg.nside}.fits"),
            title=r"Doppler $b$ (lensed kSZ)",
            unit="$b$",
            symmetric=True,
        ),
        dict(
            m=load_for_viz(OUT / "ksz" / f"kSZ_deltaT_nside{cfg.nside}.fits"),
            title=r"kSZ $\Delta T$ (all frequencies)",
            unit=r"$\mu\mathrm{K}_\mathrm{CMB}$",
            symmetric=True,
        ),
    ],
    nrows=1,
    ncols=2,
    figsize_scale=1.0,
    suptitle=rf"Kinetic SZ ($N_{{\mathrm{{side}}}}={NSIDE_VIZ}$)",
    save_name="components_ksz_mollview",
)

### 6.4 CIB — $\Delta T(\nu)$ at six Planck bands

CIB is positive-definite on average; colour limits use the 1st–99th
percentile at each frequency.

In [ ]:
freqs = list(cfg.frequencies)
mollview_gallery(
    panels=[
        dict(
            m=load_for_viz(OUT / "cib" / f"CIB_deltaT_{nu:.0f}GHz_nside{cfg.nside}.fits"),
            title=rf"CIB $\Delta T$ — ${nu:.0f}\,\mathrm{{GHz}}$"
            + ("" if nu in (217, 353, 545, 857) else " (SED approx.)"),
            unit=r"$\mu\mathrm{K}_\mathrm{CMB}$",
            symmetric=False,
            pct=(1, 99),
            cmap="viridis",
        )
        for nu in freqs
    ],
    nrows=2,
    ncols=3,
    figsize_scale=1.0,
    suptitle=rf"CIB temperature maps (beam-unconvolved, $N_{{\mathrm{{side}}}}={NSIDE_VIZ}$)",
    save_name="components_cib_deltaT_allfreq_mollview",
)

### 6.5 Overview — one panel per component (353 GHz)

In [ ]:
nu_ref = 353.0
mollview_gallery(
    panels=[
        dict(
            m=load_for_viz(OUT / "cmb" / f"primary_CMB_T_lensed_nside{cfg.nside}_seed{cfg.seed}.fits"),
            title="CMB",
            unit=r"$\mu\mathrm{K}_\mathrm{CMB}$",
            symmetric=True,
        ),
        dict(
            m=load_for_viz(OUT / "tsz" / f"tSZ_deltaT_{nu_ref:.0f}GHz_nside{cfg.nside}.fits"),
            title=rf"tSZ ${nu_ref:.0f}\,\mathrm{{GHz}}$",
            unit=r"$\mu\mathrm{K}_\mathrm{CMB}$",
            symmetric=True,
        ),
        dict(
            m=load_for_viz(OUT / "ksz" / f"kSZ_deltaT_nside{cfg.nside}.fits"),
            title="kSZ",
            unit=r"$\mu\mathrm{K}_\mathrm{CMB}$",
            symmetric=True,
        ),
        dict(
            m=load_for_viz(OUT / "cib" / f"CIB_deltaT_{nu_ref:.0f}GHz_nside{cfg.nside}.fits"),
            title=rf"CIB ${nu_ref:.0f}\,\mathrm{{GHz}}$",
            unit=r"$\mu\mathrm{K}_\mathrm{CMB}$",
            symmetric=False,
            cmap="viridis",
        ),
    ],
    nrows=1,
    ncols=4,
    figsize_scale=1.0,
    suptitle=rf"Component overview at ${nu_ref:.0f}\,\mathrm{{GHz}}$ "
    rf"($N_{{\mathrm{{side}}}}={NSIDE_VIZ}$, beam-unconvolved)",
    save_name="components_overview_353GHz_mollview",
)